# Compare Greedy and Dynamic Programming

Compare two 0/1-knapsack solvers on the same instance:
- Greedy: make a local decision by considering items in descending value-to-weight ratio.
- Dynamic programming (DP): examine combinations through smaller subproblems and returns an optimal solution.

In [12]:
from knapsack_ml_experiment import (
    Item,
    solve_dynamic_programming,
    solve_greedy,
)

## Define one knapsack instance

The capacity is 7. Greedy will rank the items as A, B, C because their value-to-weight ratios are 3.0, 2.67, and 2.25, respectively.

In [13]:
items = [
    Item("A", weight=2, value=6),
    Item("B", weight=3, value=8),
    Item("C", weight=4, value=9),
]
capacity = 7

for item in items:
    print(
        f"{item.name}: weight={item.weight}, value={item.value}, "
        f"ratio={item.value_to_weight_ratio:.2f}"
    )

A: weight=2, value=6, ratio=3.00
B: weight=3, value=8, ratio=2.67
C: weight=4, value=9, ratio=2.25


## Run both solvers

Both functions receive the identical items and capacity, so the only difference is their solution strategy.

In [14]:
greedy_solution = solve_greedy(items, capacity)
dp_solution = solve_dynamic_programming(items, capacity)

In [15]:
def summarize(solution):
    return {
        "selected_items": [item.name for item in solution.selected_items],
        "total_weight": solution.total_weight,
        "total_value": solution.total_value,
    }

print("Greedy:", summarize(greedy_solution))
print("DP:    ", summarize(dp_solution))

Greedy: {'selected_items': ['A', 'B'], 'total_weight': 5, 'total_value': 14}
DP:     {'selected_items': ['B', 'C'], 'total_weight': 7, 'total_value': 17}


## Compare solution quality

We treat the exact DP value as the optimum. The absolute gap measures how much value greedy loses. The relative gap expresses that loss as a fraction of the optimal value.

In [16]:
absolute_gap = dp_solution.total_value - greedy_solution.total_value
relative_gap = absolute_gap / dp_solution.total_value
greedy_failed = greedy_solution.total_value < dp_solution.total_value

print(f"Did greedy fail? {greedy_failed}")
print(f"Absolute gap: {absolute_gap}")
print(f"Relative gap: {relative_gap:.2%}")

Did greedy fail? True
Absolute gap: 3
Relative gap: 17.65%


## Interpretation

Greedy first selects A and then B, producing weight 5 and value 14. Item C can no longer fit because only 2 units of capacity remain.

DP instead selects B and C, which exactly fill the capacity and produce value 17. Greedy therefore loses 3 units of value, giving a relative gap of about 17.65%.

Choosing the best ratio at each step does not account for how the selected items interact with remaining capacity. Thus, ratio-greedy may not the the most optimal method for 0/1 knapsack problem.

# Validate the Experimental Foundation

Before using DP to label training data, we need confidence that both solvers follow the same contract. Every returned solution must:

- stay within the capacity;
- report totals that match its selected items;
- select only items from the original instance; and
- select each available item at most once.

We will check those properties across several manually understood cases.

In [17]:
from collections import Counter

import pandas as pd

In [18]:
def validate_solution(items, capacity, solution):
    totals_are_consistent = (
        solution.total_weight
        == sum(item.weight for item in solution.selected_items)
        and solution.total_value
        == sum(item.value for item in solution.selected_items)
    )
    uses_only_available_items = not (
        Counter(solution.selected_items) - Counter(items)
    )

    return {
        "feasible": solution.total_weight <= capacity,
        "totals_consistent": totals_are_consistent,
        "uses_only_available_items": uses_only_available_items,
    }

## Check several kinds of instances

The cases cover an instance where greedy is optimal, a counterexample where it fails, an item that cannot fit, and an empty item list.

In [19]:
validation_cases = {
    "greedy_is_optimal": (
        [
            Item("A", weight=1, value=4),
            Item("B", weight=2, value=6),
            Item("C", weight=3, value=6),
        ],
        3,
    ),
    "greedy_counterexample": (
        [
            Item("A", weight=10, value=60),
            Item("B", weight=20, value=100),
            Item("C", weight=30, value=120),
        ],
        50,
    ),
    "item_too_heavy": ([Item("A", weight=11, value=100)], 10),
    "empty_items": ([], 10),
}

In [20]:
solvers = {
    "greedy": solve_greedy,
    "dynamic_programming": solve_dynamic_programming,
}
records = []

for case_name, (case_items, case_capacity) in validation_cases.items():
    for solver_name, solver in solvers.items():
        solution = solver(case_items, case_capacity)
        checks = validate_solution(case_items, case_capacity, solution)

        # A failed assertion means the solver broke our shared contract.
        assert all(checks.values())

        records.append({
            "case": case_name,
            "solver": solver_name,
            "selected_items": ", ".join(
                item.name for item in solution.selected_items
            ) or "none",
            "total_weight": solution.total_weight,
            "total_value": solution.total_value,
            **checks,
        })

temp_results = pd.DataFrame(records)
temp_results

,case,solver,selected_items,total_weight,total_value,feasible,totals_consistent,uses_only_available_items
0,greedy_is_optimal,greedy,"A, B",3,10,True,True,True
1,greedy_is_optimal,dynamic_programming,"A, B",3,10,True,True,True
2,greedy_counterexample,greedy,"A, B",30,160,True,True,True
3,greedy_counterexample,dynamic_programming,"B, C",50,220,True,True,True
4,item_too_heavy,greedy,none,0,0,True,True,True
5,item_too_heavy,dynamic_programming,none,0,0,True,True,True
6,empty_items,greedy,none,0,0,True,True,True
7,empty_items,dynamic_programming,none,0,0,True,True,True


## Interpretation

Both solvers return feasible, internally consistent solutions for every checked case. They agree when ratio-greedy happens to be optimal and safely return empty solutions when nothing can be selected. On the classic counterexample, greedy returns value 160 while DP returns 220.

These checks do not prove that the implementations are correct for every possible input, but they give us a tested foundation. We can now use DP as the exact reference when generating labels for the next phase. The reusable versions of these checks live in the project's unit tests.